# Phase 4: graph analytics (Cypher + GDS), raw vs resolved

Runs the `analytics` module on both graph versions in Neo4j. Every query and algorithm takes `kg_version`.
No LLM calls. Figures go to `docs/figures/`, the RQ2 table to `eval/results/`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from analytics import get_driver, queries as Q, algorithms as A, quality as QL, KG_VERSIONS
FIG = ROOT / "docs" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 80)
d = get_driver(); Q.ensure_schema(d)

## 1. Cypher library: lookups, paths, aggregations, co-occurrence, provenance

In [ ]:
V = "resolved"
for name in ["Google DeepMind", "deepmnd", "OpenAI"]:
    print(name, "->", Q.find(d, name, V)["name"])
sam, ms, oai = (Q.find(d, n, V)["id"] for n in ("Sam Altman", "Microsoft", "OpenAI"))
display(pd.DataFrame(Q.neighbours(d, oai, V, 10)))
display(pd.DataFrame(Q.shortest_path(d, sam, ms, V)))
display(pd.DataFrame(Q.paths(d, sam, ms, V, hops=3, limit=5)))

In [ ]:
for v in KG_VERSIONS:
    print(f"--- top model developers ({v})")
    display(pd.DataFrame(Q.top_model_developers(d, v, 8)))

In [ ]:
display(pd.DataFrame(Q.top_technologies(d, V, 8)))
display(pd.DataFrame(Q.top_papers(d, V, 8)))
display(pd.DataFrame(Q.top_affiliations(d, V, 8)))
display(pd.DataFrame(Q.top_by_documents(d, V, 8)))
display(pd.DataFrame(Q.cooccurrence(d, V, 3, 10)))

In [ ]:
e = Q.sample_edges(d, "DEVELOPED_BY", V, 1)[0]
display(pd.DataFrame(Q.edge_provenance(d, e["source"], "DEVELOPED_BY", e["target"], V))[["source", "relation", "target", "evidence", "confidence", "url"]])

## 2. GDS algorithms (PageRank, degree, betweenness, Louvain, node similarity, WCC), written back as `<algorithm>_<kg_version>`

In [ ]:
runs = {v: A.run_all(d, v) for v in KG_VERSIONS}
pd.DataFrame({v: {"entities": r["projection_directed"]["nodes"], "edges": r["projection_directed"]["relationships"],
                  "pagerank iterations": r["pagerank"]["ranIterations"], "louvain communities": r["louvain"]["communityCount"],
                  "modularity": round(r["louvain"]["modularity"], 3), "components": r["wcc"]["componentCount"],
                  "largest component": r["lcc"]["size"], "largest component %": r["lcc"]["pct"],
                  "entities with similar": r["node_similarity"]["entities_with_similar"], "seconds": r["seconds"]}
              for v, r in runs.items()})

In [ ]:
for v in KG_VERSIONS:
    print(f"--- {v}: top PageRank / betweenness / largest communities")
    display(pd.DataFrame(A.top(d, "pagerank", v, 10)))
    display(pd.DataFrame(A.top(d, "betweenness", v, 5)))
    display(pd.DataFrame(A.communities(d, v, 6)))
dm = Q.find(d, "DeepMind", V)["id"]
display(pd.DataFrame(A.similar(d, dm, V)))

## 3. Figures

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for v, m in (("raw", "o"), ("resolved", "s")):
    s = pd.Series(A.degrees(d, v)); s = s[s > 0].value_counts().sort_index()
    ax.loglog(s.index, s.values, m, alpha=0.7, label=v)
ax.set_xlabel("degree (relation edges)"); ax.set_ylabel("number of entities")
ax.set_title("Degree distribution (log-log, degree > 0)"); ax.legend()
fig.tight_layout(); fig.savefig(FIG / "phase4_degree_distribution.png", dpi=150); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharex=True)
for ax, v in zip(axes, KG_VERSIONS):
    t = pd.DataFrame(A.top(d, "pagerank", v, 15)).iloc[::-1]
    ax.barh(t["name"] + " (" + t["label"] + ")", t["pagerank"], color="#4C72B0" if v == "raw" else "#55A868")
    ax.set_title(f"Top-15 PageRank ({v})"); ax.set_xlabel("PageRank")
fig.tight_layout(); fig.savefig(FIG / "phase4_top15_pagerank.png", dpi=150); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
for v in KG_VERSIONS:
    sizes = A.community_sizes(d, v)
    ax.loglog(range(1, len(sizes) + 1), sizes, label=f"{v} ({sum(1 for s in sizes if s >= 2)} with size >= 2)")
ax.set_xlabel("community rank"); ax.set_ylabel("size (entities)"); ax.set_title("Louvain community sizes")
ax.legend(); fig.tight_layout(); fig.savefig(FIG / "phase4_community_sizes.png", dpi=150); plt.show()

## 4. RQ2: raw vs resolved graph quality

In [ ]:
rq2 = QL.compare(d)
csv_path, md_path = QL.save(rq2, str(ROOT / "eval" / "results"))
table = pd.DataFrame(QL.table_rows(rq2))[["metric", "raw_fmt", "resolved_fmt", "change_fmt"]]
table.columns = ["metric", "raw", "resolved", "change"]
display(table)
display(pd.DataFrame(rq2["pagerank_top10"]))
display(pd.DataFrame(rq2["resolved"]["cross_type"][:20])[["normalized_name", "labels", "names", "mentions"]])

In [ ]:
rows = [r for r in QL.table_rows(rq2) if r["change"] is not None]
fig, ax = plt.subplots(figsize=(8, 5))
vals = [100 * r["change"] for r in rows]
ax.barh([r["metric"] for r in rows], vals, color=["#C44E52" if v < 0 else "#55A868" for v in vals])
for i, (r, v) in enumerate(zip(rows, vals)):
    ax.text(v, i, f" {r['raw_fmt']} -> {r['resolved_fmt']} ", va="center", ha="left" if v >= 0 else "right", fontsize=8)
ax.axvline(0, color="black", lw=0.8); ax.invert_yaxis(); ax.set_xlim(-170, 60)
ax.set_xlabel("change from raw to resolved (%)"); ax.set_title("RQ2: what entity resolution changes")
fig.tight_layout(); fig.savefig(FIG / "phase4_rq2_comparison.png", dpi=150); plt.show()
print("saved", csv_path, md_path)
d.close()
print("ANALYTICS_NOTEBOOK_OK")